In [ ]:
!apt-get update -qq > /dev/null
!apt-get install openjdk-8-jdk-headless -qq > /dev/null


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import os

# Define Spark version
spark_version = "3.5.3"

# Download Spark (pre-built with Hadoop 3)
!wget -q https://archive.apache.org/dist/spark/spark-{spark_version}/spark-{spark_version}-bin-hadoop3.tgz

# Extract it to /usr/lib/
!tar xf spark-{spark_version}-bin-hadoop3.tgz -C /usr/lib/

# Set environment variable
os.environ["SPARK_HOME"] = f"/usr/lib/spark-{spark_version}-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PATH"] += f":{os.environ['SPARK_HOME']}/bin:{os.environ['JAVA_HOME']}/bin"


In [4]:
!ls -lh spark-3.5.3-bin-hadoop3.tgz


-rw-r--r-- 1 root root 383M Sep  9  2024 spark-3.5.3-bin-hadoop3.tgz


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("local[*]").appName("PySparkExamples").getOrCreate()

print("✅ SparkSession created successfully!")

✅ SparkSession created successfully!


1,2

In [19]:
list_A = [1, 2, 3, 4, 5]
list_B = ['a', 'b', 'c', 'd', 'e']

df = spark.createDataFrame(zip(list_A, list_B), ["list_A", "list_B"])
df.show()

+------+------+
|list_A|list_B|
+------+------+
|     1|     a|
|     2|     b|
|     3|     c|
|     4|     d|
|     5|     e|
+------+------+



3 How to get the items not common to both list A and list B?

In [32]:
list_A = [1, 2, 3, 4, 5]
list_B = [4, 5, 6, 7, 8]

not_common = list(set(list_A) ^ set(list_B))

print( not_common)


[1, 2, 3, 6, 7, 8]


4 for the below Dataframe , find the in a column find the frequency counts of unique items

In [25]:
data = [
    ("John", "Engineer"),
    ("John", "Engineer"),
    ("Mary", "Scientist"),
    ("Bob", "Engineer"),
    ("Bob", "Engineer"),
    ("Bob", "Scientist"),
    ("Sam", "Doctor")
]
columns = ["name", "job"]

df4 = spark.createDataFrame(data, columns)
df4.groupBy("job").count().orderBy(F.desc("count")).show()

+---------+-----+
|      job|count|
+---------+-----+
| Engineer|    4|
|Scientist|    2|
|   Doctor|    1|
+---------+-----+



5 keep only top 2 most frequent values as it is and replace everything else as ‘Other’?


In [26]:
job_counts = df4.groupBy("job").count().orderBy(F.desc("count"))
top_jobs = [row['job'] for row in job_counts.take(2)]

df5 = df4.withColumn(
    "job",
    F.when(F.col("job").isin(top_jobs), F.col("job")).otherwise(F.lit("Other"))
)
df5.show()

+----+---------+
|name|      job|
+----+---------+
|John| Engineer|
|John| Engineer|
|Mary|Scientist|
| Bob| Engineer|
| Bob| Engineer|
| Bob|Scientist|
| Sam|    Other|
+----+---------+



6 rename columns of a PySpark DataFrame using two lists – one containing the old column names and the other containing the new column names?

In [27]:
old_names = ["col1", "col2", "col3"]
new_names = ["new_col1", "new_col2", "new_col3"]

data = [(1, 2, 3), (4, 5, 6)]
df6 = spark.createDataFrame(data, old_names)

for old, new in zip(old_names, new_names):
    df6 = df6.withColumnRenamed(old, new)

df6.show()

+--------+--------+--------+
|new_col1|new_col2|new_col3|
+--------+--------+--------+
|       1|       2|       3|
|       4|       5|       6|
+--------+--------+--------+



7 find the numbers that are multiples of 3 from a column?

In [15]:
data = [
    (0, 7), (1, 6), (2, 9), (3, 7), (4, 3),
    (5, 8), (6, 9), (7, 8), (8, 3), (9, 8)
]
columns = ["id", "random"]
df7 = spark.createDataFrame(data, columns)

df7 = df7.withColumn("is_multiple_of_3", (F.col("random") % 3 == 0).cast("int"))
df7.show()

+---+------+----------------+
| id|random|is_multiple_of_3|
+---+------+----------------+
|  0|     7|               0|
|  1|     6|               1|
|  2|     9|               1|
|  3|     7|               0|
|  4|     3|               1|
|  5|     8|               0|
|  6|     9|               1|
|  7|     8|               0|
|  8|     3|               1|
|  9|     8|               0|
+---+------+----------------+



8 First lettter should be the capital for every word


In [35]:
data = [("john",), ("alice",), ("bob",)]
df8 = spark.createDataFrame(data, ["name"])
df8 = df8.withColumn("Name", F.initcap("name"))
df8.show()

+-----+------------+
| name|Capital_Name|
+-----+------------+
| john|        John|
|alice|       Alice|
|  bob|         Bob|
+-----+------------+



9  How to calculate the number of characters in each word in a column?

In [29]:
df9 = spark.createDataFrame(data, ["name"])
df9 = df9.withColumn("word_length", F.length("name"))
df9.show()

+-----+-----------+
| name|word_length|
+-----+-----------+
| john|          4|
|alice|          5|
|  bob|          3|
+-----+-----------+



10 Finding the null


In [36]:
data = [
    ("A", 1, None),
    ("B", None, 123),
    ("B", 3, 456),
    ("D", None, None)
]
columns = ["Name", "Value", "id"]

df10 = spark.createDataFrame(data, columns)

null_counts = {col: df10.filter(F.col(col).isNull()).count() for col in df10.columns}
print(null_counts)


{'Name': 0, 'Value': 2, 'id': 2}
